In [1]:
import time
import cv2
import numpy as np

from sdlarch_rl.utils.utils import FrameSkip, RealExcludeButtonsWrapper, AugmentObservation
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from gymnasium.wrappers.frame_stack import FrameStack
from pathlib import Path
import pygame
import gymnasium as gym

from imitation.data import serialize
from imitation.data.types import Trajectory
import torch as th
import os
import re
from sdlarch_rl import make

demo_path = 'demos-gt3/'

os.makedirs(demo_path, exist_ok=True)

last_index = -1

for p in Path(demo_path).iterdir():
    m = re.search(r"demos(\d+)\.pt$", p.name)
    if m:
        num = int(m.group(1))
        last_index = max(last_index, num)

print("last_index: " + str(last_index))

global myEnv

def make_env():
    global myEnv
    myEnv = make(
        "GranTurismo3-Ps2", 
        statename="middle_field"
        # render_mode="human"
    )
    
    buttons = myEnv.unwrapped.buttons
    print(buttons)
    to_exclude = ["UP", "DOWN", "START", "SELECT", "R1", "L1", "L2", "R2", "L3", "R3", "A"]

    print(np.setdiff1d(buttons, to_exclude))

    # env = ExcludeButtonsWrapper(myEnv, buttons, to_exclude)
    env = RealExcludeButtonsWrapper(myEnv, buttons, to_exclude)
    env = AugmentObservation(env)
    env = WarpFrame(env, width=96, height=96)
    env = FrameSkip(env, skip=4)

    return env

env = make_env()
env = FrameStack(env, 4)

print(env.action_space)

SCREEN_WIDTH = 640*3
SCREEN_HEIGHT = 480*3

pygame.init()

window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

obs, info = env.reset()

is_running = False
is_recording = False

color=(0, 0, 255)

count_record = 0

trajectories = []
# Data to store
recorded_obs = []
recorded_actions = []

pygame.joystick.init()
joysticks = [pygame.joystick.Joystick(x) for x in range(pygame.joystick.get_count())]

print("joysticks count: ", len(joysticks))

paused=False

buttons = None
joystick=None
for j in joysticks:
    name = j.get_name()
    if name not in "PS4 Controller":
        joystick=j
        buttons = j.get_numbuttons()
        break

last_k_press_time = 0
last_p_press_time = 0
last_r_press_time = 0
debounce_time = 0.3  # 300ms

while True:
    current_time = time.time()
    # pygame.event.pump()
    
    action = np.zeros(16, dtype=np.uint8)

    keys = pygame.key.get_pressed()

    # if keys[pygame.K_s]:
    #     is_running = not is_running
        
    #     time.sleep(0.5)

    #     print("k was pressed ", is_running)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_k:
                if current_time - last_k_press_time > debounce_time:
                    is_running = not is_running
                    last_k_press_time = current_time
                    print(f"K pressed. Recording: {is_running}")
            if event.key == pygame.K_p:
                if current_time - last_p_press_time > debounce_time:
                    last_p_press_time = current_time
                    paused = not paused
                    print(f"P pressed. Pausing: {paused}")
            if event.key == pygame.K_r:
                if current_time - last_r_press_time > debounce_time:
                    last_r_press_time = current_time
                    obs, info = env.reset()
                    print(f"R pressed. Reseting!!")
            elif event.key == pygame.K_ESCAPE:
                running = False

    if is_running:
        color=(0, 255, 0)
    else:
        color=(0, 0, 255)

    # controller
    for i in range(buttons):
        button = joystick.get_button(i)

        if button > 0:
            if i == 0:
                action[8] = button # A
            if i == 1:
                action[0] = button # B acel
            if i == 2:
                action[9] = button # X
            if i == 3:
                action[1] = button  # Y break
            if i == 8:
                action[14] = button # L3
            if i == 9:
                action[15] = button # R3
            if i == 7:
                action[3] = button # START
            if i == 6:
                action[2] = button # SELECT/Back
            if i == 4:
                action[12] = button # L2
            if i == 5:
                action[13] = button # R2
    hat_x, hat_y = joystick.get_hat(0)

    # Hat/D-pad
    if hat_y == 1:
        action[4] = 1
    elif hat_y == -1:
        action[5] = 1
    if hat_x == -1:
        action[6] = 1
    elif hat_x == 1:
        action[7] = 1

    # Analogs
    left_x  = joystick.get_axis(0)
    left_y  = joystick.get_axis(1)
    right_x = joystick.get_axis(2)
    right_y = joystick.get_axis(3)
    lt      = joystick.get_axis(4)
    rt      = joystick.get_axis(5)

    if left_y < -0.5:
        action[4] = 1
    elif left_y > 0.5:
        action[5] = 1

    if left_x < -0.5:
        action[6] = 1
    elif left_x > 0.5:
        action[7] = 1

    if lt > 0.5:
        action[10] =1 #
    if rt > 0.5:
        action[11] =1 #

    # change current action
    # ['B' 'LEFT' 'RIGHT' 'X' 'Y']
    # index_list:  [0, 1, 6, 7, 9]
    # ['B', 'Y', 'SELECT', 'START', 'UP', 'DOWN', 'LEFT', 'RIGHT', 'A', 'X', 'L1', 'R1', 'L2', 'R2', 'L3', 'R3']
    n_action = np.zeros(5, dtype=np.uint8)
    n_action[0] = action[0]
    n_action[1] = action[1]
    n_action[2] = action[6]
    n_action[3] = action[7]
    n_action[4] = action[9]

    action = n_action

    # else:
    #     action[0] = np.random.choice([0, 1], size=7)
    if is_running != is_recording:
        if is_running:
            print("first obs")
            recorded_obs.append(obs)

    if not paused:
        obs, reward, _, _, _ = env.step(action)

    img = myEnv.render()

    text_to_display = "Count: " + str(count_record)

    if paused:
        text_to_display = "Paused!!!!"

    if img is not None and not paused:
        img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        
        cv2.circle(img, center=(100, 100), radius=50, color=color, thickness=2)
        
        position = (50, 250)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.5
        thickness = 2
        line_type = cv2.LINE_AA


        cv2.putText(img, text_to_display, position, font, font_scale, color, thickness, line_type)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        
        
        surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))

        window.blit(surface, (0, 0))
        
    # pygame.display.update()
    pygame.display.flip()
    # clock.tick(30)
    time.sleep(1/30)

    if not paused:
        if is_running != is_recording:
            if not is_running:
                print("end recording, increase count")
                count_record = count_record + 1
    
                print("recorded_obs shape", np.array(recorded_obs).shape)
                print("recorded_actions shape", np.array(recorded_actions).shape)
    
                trajectories.append(
                    Trajectory(
                        obs=recorded_obs, 
                        acts=np.array(recorded_actions), 
                        infos = None, 
                        terminal = False,
                    )
                )
    
                recorded_obs = []
                recorded_actions = []
    
        is_recording = is_running
    
        if is_running:
            recorded_obs.append(obs)
            recorded_actions.append(action)
    
        if count_record > 5:
            print("end recording")
            break

print(len(trajectories))

th.save(trajectories, demo_path + f"demos{last_index + 1}.pt")

pygame.quit()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


last_index: 7
['B', 'Y', 'SELECT', 'START', 'UP', 'DOWN', 'LEFT', 'RIGHT', 'A', 'X', 'L1', 'R1', 'L2', 'R2', 'L3', 'R3']
['B' 'LEFT' 'RIGHT' 'X' 'Y']
index_list:  [0, 1, 6, 7, 9]
index_to_remove:  [4, 5, 3, 2, 11, 10, 12, 13, 14, 15, 8]
MultiBinary(5)
joysticks count:  1
R pressed. Reseting!!
K pressed. Recording: True
first obs
K pressed. Recording: False
end recording, increase count
recorded_obs shape (1677, 4, 96, 96, 1)
recorded_actions shape (1676, 5)
R pressed. Reseting!!
K pressed. Recording: True
first obs
K pressed. Recording: False
end recording, increase count
recorded_obs shape (1444, 4, 96, 96, 1)
recorded_actions shape (1443, 5)
R pressed. Reseting!!
K pressed. Recording: True
first obs
K pressed. Recording: False
end recording, increase count
recorded_obs shape (1046, 4, 96, 96, 1)
recorded_actions shape (1045, 5)
K pressed. Recording: True
first obs
K pressed. Recording: False
end recording, increase count
recorded_obs shape (2446, 4, 96, 96, 1)
recorded_actions shape 